<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/plongement_digits_358.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import time
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL vs DEUX NAPPES  {P_sep=0} ∪ {P_sep=ε}
#
# Question : les deux nappes S'ÉLOIGNENT-elles dans le plongement ?
# Protocole :
#   1. cloud 50/50 sur les deux nappes (P_sep renormalisé std=1, Gauss-Newton) ;
#   2. diagnostic de séparation AVANT (espace ambiant) ;
#   3. classification QP (semi-interpolation Sobolev, expert polynomial) AVANT ;
#   4. plongement spectral (projections aléatoires + base polynomiale + Gram
#      Sobolev, vecteurs propres normalisés) — le programme historique ;
#   5. diagnostic de séparation APRÈS (espace plongé) ;
#   6. classification QP APRÈS (petit degré : si le plongement redresse, un
#      degré bas doit suffire) ;
#   7. baseline Ridge poly avant/après (RBF supprimé).
# ══════════════════════════════════════════════════════════════════════════════

params = {
    # ── DONNÉES (digits sklearn, 8×8 -> moyenné en 4×4 -> dim 16, 3 vs 8) ─────
    "digits_neg":  [3, 5],      # classe -1 : UNION de ces chiffres
    "digits_pos":  [8],         # classe +1 : UNION de ces chiffres
    "n_train":      5,         # points étiquetés (5 par classe)
    "n_test":       200,        # points de test (au max ~180 par classe pour 3 vs 8)
    "n_unlabeled":  1000,        # points non étiquetés (le RESTE du dataset digits)

    # ── NORME DE SOBOLEV (classification) ─────────────────────────────────────
    "weights":      {0: 1e-4, 1: 1, 2: 0, 3: 0},   # dim 16 : w2,w3 impraticables
    "weights_emb":  {0: 1e-4, 1: 1, 2: 0, 3: 0},   # dim 30 après plongement

    # ── CLASSIFICATION QP (semi-interpolation) ────────────────────────────────
    "qp_margin":     1.0,
    "deg_clf_avant": 3,
    "deg_clf_apres": 2,
    "thres1":        1e-15,
    "thres2":        1e3,
    "const_pen":     0,
    "n_G":           1500,

    # ── PLONGEMENT (avec projections aléatoires) ──────────────────────────────
    "n_layers":     1,
    "embed_dim":    70,
    "lambda_G":     1.0,
    "k_proj":       16,          # projections en dim 8 : deux blocs orthogonaux
                                #   -> monômes en dim 8 par bloc (rapide)
    "emb_thres1":   0.9,        # bande [plancher, plafond] ; on garde les DERNIERS
    "emb_thres2":   4,        #   embed_dim disponibles dans la bande
    "n_deg":        2,          # degré des monômes par bloc (dim 8 -> C(14,6)=3003)
    "activation":   "none",
    "weights_plg":  {0: 0, 1: 1, 2: 0, 3: 0},   # w1 seul
    "seed":         42,
}

# ══════════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES DONNÉES : digits sklearn, chiffres a vs b, réduits en 4×4
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.datasets import load_digits

def load_digits_binary(neg_list, pos_list, n_train, n_test, n_unlabeled, seed=42):
    """Charge digits sklearn. Classe -1 = union des chiffres dans neg_list ;
    classe +1 = union des chiffres dans pos_list. 8×8 -> 4×4 par moyenne 2×2.
    Split équilibré (autant de chaque classe dans train/test)."""
    D = load_digits()
    neg_set = set(neg_list); pos_set = set(pos_list)
    mask = np.isin(D.target, list(neg_set | pos_set))
    X = D.images[mask].astype(float)
    tgt = D.target[mask]
    y = np.where(np.isin(tgt, list(pos_set)), 1.0, -1.0)
    X4 = X.reshape(-1, 4, 2, 4, 2).mean(axis=(2, 4)).reshape(-1, 16)
    X4 = (X4 - X4.mean(0)) / (X4.std(0) + 1e-8)
    rng = np.random.default_rng(seed)
    idx_neg = np.where(y < 0)[0]; idx_pos = np.where(y > 0)[0]
    rng.shuffle(idx_neg); rng.shuffle(idx_pos)
    n_tr_c = n_train // 2; n_te_c = n_test // 2
    tr = np.concatenate([idx_neg[:n_tr_c], idx_pos[:n_tr_c]])
    te = np.concatenate([idx_neg[n_tr_c:n_tr_c+n_te_c],
                         idx_pos[n_tr_c:n_tr_c+n_te_c]])
    used = set(tr) | set(te)
    unlab = np.array([i for i in range(len(X4)) if i not in used])
    if len(unlab) > n_unlabeled:
        unlab = rng.choice(unlab, n_unlabeled, replace=False)
    rng.shuffle(tr); rng.shuffle(te); rng.shuffle(unlab)
    return X4[tr], y[tr], X4[te], y[te], X4[unlab], y[unlab]

X_train, y_train, X_test, y_test, X_unlab, y_unlab = load_digits_binary(
    params["digits_neg"], params["digits_pos"],
    params["n_train"], params["n_test"], params["n_unlabeled"],
    seed=params["seed"])
d = X_train.shape[1]
X_all = np.vstack([X_train, X_unlab]); N_all = len(X_all)
y_all = np.concatenate([y_train, y_unlab])
_neg_s = "{" + ",".join(map(str, params["digits_neg"])) + "}"
_pos_s = "{" + ",".join(map(str, params["digits_pos"])) + "}"
print(f"Digits {_neg_s} vs {_pos_s}, 4×4 -> dim {d}")
print(f"  train    : {len(X_train)} pts  ({int((y_train<0).sum())} × {_neg_s}, "
      f"{int((y_train>0).sum())} × {_pos_s})")
print(f"  test     : {len(X_test)} pts  ({int((y_test<0).sum())} × {_neg_s}, "
      f"{int((y_test>0).sum())} × {_pos_s})")
print(f"  unlabeled: {len(X_unlab)} pts  (labels cachés pour le fit)")

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC DE SÉPARATION  (avant / après plongement)
#   ratio inter/intra : distance moyenne entre nappes / distance moyenne au sein
#   d'une nappe. > 1 = les nappes s'éloignent. On mesure sur le cloud (labels de
#   génération, diagnostic pur).
# ══════════════════════════════════════════════════════════════════════════════
def separation_diag(Z, y, n_sub=800, seed=0, titre=""):
    rng = np.random.default_rng(seed)
    i0 = np.where(y < 0)[0]; i1 = np.where(y > 0)[0]
    i0 = rng.choice(i0, min(n_sub, len(i0)), replace=False)
    i1 = rng.choice(i1, min(n_sub, len(i1)), replace=False)
    Z0, Z1 = Z[i0], Z[i1]
    def mdist(A, B):
        return np.mean(np.sqrt(np.maximum(
            np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T, 0)))
    intra = 0.5*(mdist(Z0, Z0) + mdist(Z1, Z1))
    inter = mdist(Z0, Z1)
    # 1-NN inter-nappe : fraction de points dont le plus proche voisin est de
    # l'AUTRE nappe (0 = nappes bien séparées, ~0.5 = mélangées)
    Zs = np.vstack([Z0, Z1]); ys = np.concatenate([-np.ones(len(Z0)), np.ones(len(Z1))])
    D = np.sum(Zs**2,1)[:,None] + np.sum(Zs**2,1)[None,:] - 2*Zs@Zs.T
    np.fill_diagonal(D, np.inf)
    nn = np.argmin(D, axis=1)
    frac_cross = np.mean(ys[nn] != ys)
    print(f"  [{titre}] inter/intra = {inter/intra:.4f}   "
          f"(inter={inter:.3f} intra={intra:.3f})   1-NN croisé = {frac_cross:.3f}")
    return inter/intra, frac_cross

# ══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATEUR QP  (semi-interpolation Sobolev, expert polynomial générique)
#   min ‖u‖²_H  s.c.  y_i·u(x_i) >= marge,  base H-orthonormée + constante libre.
#   Générique : X de dimension quelconque (ambiant OU plongé).
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin, verbose=False):
    n, k = A.shape
    Gr = G + 1e-12*np.eye(k)
    con = LinearConstraint(np.diag(y)@A, lb=margin, ub=np.inf)
    try:    c0 = np.linalg.lstsq(A, 1.5*margin*y, rcond=None)[0]
    except Exception: c0 = np.zeros(k)
    res = minimize(lambda c: c@Gr@c, c0, jac=lambda c: 2*Gr@c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y*(A@c)))
    return c, marge_eff >= margin - 1e-4, marge_eff

def classify_poly_qp(X_tr, y_tr, X_te, y_te, X_cloud, deg, weights, titre=""):
    """Classifieur polynomial par semi-interpolation QP. Retourne (acc, auc)."""
    n, dd = X_tr.shape
    # normalisation par coordonnée -> [-1,1] (calculée sur le cloud)
    lo = X_cloud.min(axis=0); hi = X_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(X): return 2*(X - lo)/span - 1.0
    SC = 2.0/span                                      # ∂u/∂x par coordonnée

    poly = PolynomialFeatures(degree=deg, include_bias=False)
    Phi_tr = poly.fit_transform(to_u(X_tr))
    powers = poly.powers_; n_feat = Phi_tr.shape[1]

    def pderiv(U, coefs, dims=()):
        P = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m*P[:, dm]; P[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :]**P[None, v, :]).prod(2)@m[v]

    rng = np.random.default_rng(4242)
    idx = rng.choice(len(X_cloud), min(params["n_G"], len(X_cloud)), replace=False)
    U_G = to_u(X_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.); w2 = weights.get(2, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0*(PHI.T@PHI)/nG
    if w1:
        GG = np.zeros((nG, n_feat, dd))
        for j in range(n_feat):
            for a in range(dd):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a]*pderiv(U_G, E[j], (a,))
        G += w1*np.einsum('xik,xjk->ij', GG, GG)/nG
    if w2:
        HH = np.zeros((nG, n_feat, dd, dd))
        for j in range(n_feat):
            for a in range(dd):
                for b in range(a, dd):
                    if (powers[j, a] > 0 and powers[j, b] > 0) or (a == b and powers[j, a] > 1):
                        v = SC[a]*SC[b]*pderiv(U_G, E[j], (a, b))
                        HH[:, j, a, b] = v; HH[:, j, b, a] = v
        G += w2*np.einsum('xikl,xjkl->ij', HH, HH)/nG

    # base H-orthonormée (bande) + centrage + constante libre ancrée
    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G); smax = 1 #s_g.max()
    keep = (s_g > smax*params["thres1"]) & (s_g < smax*params["thres2"])
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep]/np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi)@T, np.ones((n, 1))])
    G_qp = np.eye(r+1); G_qp[-1, -1] = params["const_pen"]
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, params["qp_margin"])
    coef = T@sol[:r]; off = sol[-1] - mean_phi@coef

    f_te = poly.transform(to_u(X_te))@coef + off
    f_tr = Phi_tr@coef + off
    def acc(f, y):
        sg = np.sign(f)
        return np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float)))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] deg={deg} feat={n_feat} rang={r} | faisable={feas} "
          f"marge={marge:.3f} ‖u‖_H={np.sqrt(max(sol[:r]@sol[:r],0)):.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.2f} acc_te={acc(f_te,y_te):.4f} AUC={auc:.4f}")
    return acc(f_te, y_te), auc

def classify_ridge(X_tr, y_tr, X_te, y_te, deg, titre=""):
    poly = PolynomialFeatures(degree=deg)
    r = Ridge(alpha=1e-8).fit(poly.fit_transform(X_tr), y_tr)
    f = r.predict(poly.transform(X_te))
    a = np.mean(np.sign(f) == np.sign(y_te))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f)
    except Exception: auc = float('nan')
    print(f"  [{titre}] Ridge deg={deg} : acc_te={a:.4f} AUC={auc:.4f}")
    return a, auc

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL  (programme historique, repris tel quel)
# ══════════════════════════════════════════════════════════════════════════════
def activate(X, mode):
    if mode == "tanh":    return np.tanh(X)
    if mode == "sigmoid": return 1/(1 + np.exp(-X))
    return X

def make_projections_ortho(dd, k_proj, rng):
    """Projections sur des sous-espaces ORTHOGONAUX deux à deux qui PARTITIONNENT
    R^dd. Construction : QR d'une matrice gaussienne dd×dd -> base orthonormée
    aléatoire ; on la découpe en blocs de k_proj lignes. Le dernier bloc a
    dimension dd mod k_proj s'il ne divise pas (choix unique : l'orthogonal des
    précédents). k_proj est PLAFONNÉ à dd (au-delà, les vecteurs propres seraient
    redondants pour beaucoup de calcul en plus).
    GARANTIE : la concaténation des bases relevées contient TOUS les polynômes de
    degré 1 dans son span — les lignes des blocs forment une base de R^dd, donc
    toute forme linéaire x ↦ v·x est combinaison des coordonnées projetées."""
    k = min(k_proj, dd)
    M = rng.standard_normal((dd, dd))
    Q, _ = np.linalg.qr(M)                     # lignes de Q.T = base orthonormée
    B = Q.T                                    # (dd, dd) : chaque ligne unitaire
    blocks = [B[i:i+k] for i in range(0, dd, k)]
    return blocks                              # liste de (k_p, dd), k_p<=k

def poly_features_projected(X, Pis, n_deg):
    """Pis : liste de matrices (k_p, d). Monômes de degré <= n_deg par projection
    (k_p peut varier -> monômes par bloc). Retourne Phi concaténée + la liste des
    monômes par bloc."""
    n = X.shape[0]
    cols = []; monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        Zp = X @ Pi_p.T                        # (n, k_p)
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes)
        Phi_p = np.ones((n, len(monomes)))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        cols.append(Phi_p)
    return np.hstack(cols), monomes_list

def poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G):
    n = X_cloud.shape[0]
    sizes = []
    monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes); sizes.append(len(monomes))
    n_funcs = sum(sizes)
    G = np.zeros((n_funcs, n_funcs))
    off = 0
    for Pi_p, monomes in zip(Pis, monomes_list):
        k_p = Pi_p.shape[0]; n_mon = len(monomes)
        sl = slice(off, off + n_mon); off += n_mon
        Zp = X_cloud @ Pi_p.T
        Phi_p = np.ones((n, n_mon))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        w0 = weights.get(0, 0.)
        if w0: G[sl, sl] += w0*(Phi_p.T@Phi_p)/n
        w1 = weights.get(1, 0.)
        if w1:
            GradZ = np.zeros((n, n_mon, k_p))
            for m, alpha in enumerate(monomes):
                for j in range(k_p):
                    if alpha[j] > 0:
                        col = np.ones(n)*alpha[j]
                        for l, e in enumerate(alpha):
                            e2 = e - 1 if l == j else e
                            if e2 > 0: col *= Zp[:, l]**e2
                        GradZ[:, m, j] = col
            GradX = np.einsum('nmj,jd->nmd', GradZ, Pi_p)
            G[sl, sl] += w1*np.einsum('nmi,nli->ml', GradX, GradX)/n
    G += lambda_G*np.eye(n_funcs)
    return G

def poly_embedding(X_input, X_cloud, weights, lambda_G,
                   k_proj, n_deg, embed_dim, rng,
                   emb_thres1, emb_thres2):
    """Plongement spectral. Projections orthogonales partitionnant l'espace
    (n_proj dérivé = ceil(d/k_proj)). Sélection des vecteurs propres par BANDE :
    on garde emb_thres1 < s/s_max < emb_thres2 (plancher ET plafond — les plus
    grandes VP sont dominées par les fonctions de grande norme H, à jeter ;
    les plus petites sont le noyau numérique). Cap à embed_dim modes."""
    Pis = make_projections_ortho(X_input.shape[1], k_proj, rng)
    print(f"  projections orthogonales : {len(Pis)} blocs de dims "
          f"{[p.shape[0] for p in Pis]} (partition de R^{X_input.shape[1]})")
    Phi_cloud, monomes = poly_features_projected(X_cloud, Pis, n_deg)
    G = poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G)
    eigvals, eigvecs = np.linalg.eigh(G)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]; eigvecs = eigvecs[:, order]
    smax = 1  # échelle absolue
    # bande [plancher, plafond] activée. On garde les embed_dim DERNIERS
    # (plus petites VP) de la bande — les directions les plus lisses sur le cloud.
    band_idx = np.where((eigvals > emb_thres1*smax) & (eigvals < emb_thres2*smax))[0]
    if len(band_idx) == 0:
        raise ValueError(f"bande vide : s ∈ [{eigvals.min():.2e}, {eigvals.max():.2e}] ; "
                         f"ajuster emb_thres1/emb_thres2")
    idx_sel = band_idx[-embed_dim:] if len(band_idx) > embed_dim else band_idx
    V = eigvecs[:, idx_sel]; L = eigvals[idx_sel]
    n_above = int((eigvals >= emb_thres2*smax).sum())
    n_below = int((eigvals <= emb_thres1*smax).sum())
    print(f"  spectre : {len(eigvals)} VP | "
          f"jetées : {n_above} en haut (s>={emb_thres2:g}), "
          f"{n_below} en bas (s<={emb_thres1:g}) | "
          f"disponibles dans la bande : {len(band_idx)} | retenues : {len(L)}")
    print(f"  VP gardées ({len(L)}) : {np.array2string(L, precision=3, max_line_width=100)}")
    V_norm = V/np.sqrt(L)[None, :]
    # centrer les coordonnées du plongement (moyenne nulle sur le cloud)
    embed_cld = Phi_cloud@V_norm
    embed_means = embed_cld.mean(axis=0)     # mémorisé pour les nouvelles données
    Phi_in, _ = poly_features_projected(X_input, Pis, n_deg)
    return Phi_in@V_norm - embed_means, Pis, V_norm, embed_means

# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
# centrage (comme le programme historique) — le plongement travaille centré
center = X_all.mean(axis=0)
Xc_train = X_train - center; Xc_test = X_test - center; Xc_all = X_all - center

print("\n" + "="*72)
print("AVANT PLONGEMENT (espace ambiant)")
print("="*72)
sep0 = separation_diag(Xc_all, y_all, titre="séparation ambiant")
acc0, auc0 = classify_poly_qp(Xc_train, y_train, Xc_test, y_test, Xc_all,
                              params["deg_clf_avant"], params["weights"],
                              titre="QP ambiant")
classify_ridge(Xc_train, y_train, Xc_test, y_test, 3, titre="baseline ambiant")

print("\n" + "="*72)
print(f"PLONGEMENT SPECTRAL ({params['n_layers']} couche(s), "
      f"embed_dim={params['embed_dim']}, k_proj={params['k_proj']} (blocs orthogonaux), "
      f"n_deg={params['n_deg']}, bande=[{params['emb_thres1']:g},{params['emb_thres2']:g}])")
print("="*72)
rng = np.random.default_rng(0)
E_train, E_test, E_all = Xc_train, Xc_test, Xc_all
for layer in range(params["n_layers"]):
    print(f"── couche {layer+1} ──")
    E_all_new, Pis, V_norm, means = poly_embedding(
        E_all, E_all, params["weights_plg"], params["lambda_G"],
        params["k_proj"], params["n_deg"],
        params["embed_dim"], rng,
        params["emb_thres1"], params["emb_thres2"])
    def embed_new(Xin, Pis=Pis, V=V_norm, m=means, nd=params["n_deg"]):
        Phi, _ = poly_features_projected(Xin, Pis, nd)
        return activate(Phi@V - m, params["activation"])
    E_train = embed_new(E_train); E_test = embed_new(E_test)
    E_all = activate(E_all_new, params["activation"])
    print(f"  dims : {E_all.shape[1]}")

print("\n" + "="*72)
print("APRÈS PLONGEMENT (espace plongé)")
print("="*72)
sep1 = separation_diag(E_all, y_all, titre="séparation plongé")
acc1, auc1 = classify_poly_qp(E_train, y_train, E_test, y_test, E_all,
                              params["deg_clf_apres"], params["weights_emb"],
                              titre="QP plongé")
classify_ridge(E_train, y_train, E_test, y_test, 2, titre="baseline plongé")

print("\n" + "="*72)
print("BILAN")
print("="*72)
print(f"  séparation inter/intra : {sep0[0]:.4f} -> {sep1[0]:.4f}   "
      f"({'ÉLOIGNEMENT' if sep1[0] > sep0[0] else 'pas d’éloignement'})")
print(f"  1-NN croisé            : {sep0[1]:.3f} -> {sep1[1]:.3f}   "
      f"(plus petit = mieux séparé)")
print(f"  QP  acc_te             : {acc0:.4f} (deg {params['deg_clf_avant']}, ambiant) "
      f"-> {acc1:.4f} (deg {params['deg_clf_apres']}, plongé)")




# ══════════════════════════════════════════════════════════════════════════════
# BASELINES : SVM RBF et Kernel Ridge RBF (comparaison à l'état de l'art)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV

print("\n" + "="*72)
print("BASELINES RBF")
print("="*72)

# petite grille — n_train=500 reste rapide en RBF
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(Xc_train, y_train)
f_svm = svm.decision_function(Xc_test)
acc_svm = np.mean(np.sign(f_svm) == np.sign(y_test))
try:    auc_svm = roc_auc_score((y_test > 0).astype(int), f_svm)
except Exception: auc_svm = float('nan')
print(f"  [SVM RBF]   best={svm.best_params_} | acc_te={acc_svm:.4f} AUC={auc_svm:.4f}")

param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(Xc_train, y_train)
f_kr = kr.predict(Xc_test)
acc_kr = np.mean(np.sign(f_kr) == np.sign(y_test))
try:    auc_kr = roc_auc_score((y_test > 0).astype(int), f_kr)
except Exception: auc_kr = float('nan')
print(f"  [Ridge RBF] best={kr.best_params_} | acc_te={acc_kr:.4f} AUC={auc_kr:.4f}")

print("\n" + "="*72)
print("BILAN COMPLET")
print("="*72)
print(f"  QP ambiant (deg {params['deg_clf_avant']}) : acc={acc0:.4f} AUC={auc0:.4f}")
#print(f"  Multicouche Sobolev         : acc={acc_ml:.4f} AUC={auc_ml:.4f}")
print(f"  Experts + Phase 2 QP        : acc={_acc(pred_test, y_test):.4f} AUC={auc_p2:.4f}")
print(f"  SVM RBF                     : acc={acc_svm:.4f} AUC={auc_svm:.4f}")
print(f"  Ridge RBF                   : acc={acc_kr:.4f} AUC={auc_kr:.4f}")

Digits {3,5} vs {8}, 4×4 -> dim 16
  train    : 4 pts  (2 × {3,5}, 2 × {8})
  test     : 200 pts  (100 × {3,5}, 100 × {8})
  unlabeled: 335 pts  (labels cachés pour le fit)

AVANT PLONGEMENT (espace ambiant)
  [séparation ambiant] inter/intra = 1.1168   (inter=5.507 intra=4.931)   1-NN croisé = 0.047
  [QP ambiant] deg=3 feat=968 rang=968 | faisable=True marge=1.000 ‖u‖_H=0.1564 | acc_tr=1.00 acc_te=0.5100 AUC=0.8824
  [baseline ambiant] Ridge deg=3 : acc_te=0.5200 AUC=0.7922

PLONGEMENT SPECTRAL (1 couche(s), embed_dim=70, k_proj=16 (blocs orthogonaux), n_deg=2, bande=[0.9,4])
── couche 1 ──
  projections orthogonales : 1 blocs de dims [16] (partition de R^16)
  spectre : 152 VP | jetées : 29 en haut (s>=4), 0 en bas (s<=0.9) | disponibles dans la bande : 123 | retenues : 70
  VP gardées (70) : [2.347 2.327 2.307 2.297 2.286 2.268 2.262 2.253 2.218 2.213 2.192 2.174 2.159 2.144 2.126 2.11
 2.087 2.07  2.06  2.035 2.019 2.015 2.    2.    2.    2.    2.    2.    2.    2.    2.    2.
 2.

# Nouvelle section

In [ ]:
from google.colab import drive
drive.mount('/content/drive')